# role_confirm

> Phase 24 (`specs/multiagent/phase24/`, experiment C1c): confirm the dedicated bridge. C1b's Gate C1b asked (2026-11-12): `bridgeFix` was no more effective than `bridge1`, while the dedicated role alone (`bridgeRole`) was the best coupled arm on 5 exploratory boards. C1c tests `bridgeRole` against `bridge1` on 8 fresh paired seeds (1810-1817). `ARMS24`, `c1cGrid` and `c1cRange`; `C1C_PLAN`, frozen after the user approved the design and its $10.50-11.03 estimate; `gateC1c` (part i, effective; part ii, safe), `c1cSummary` and `writeC1c`.

In [ ]:
#| default_exp role_confirm

In [ ]:
#| hide
from nbdev.showdoc import *

## Grid and cost

Two arms from `ARMS23`, unchanged: C1's `bridge1` (a solver bridge per squad pair) and C1b's `bridgeRole` (a dedicated bridge without evidence). 64 agents, independent liars, seeds 1810-1817 (fresh: C1 and C1b used 1800-1807 and the pilot seed 1890). The cost per board is the measured mean over C1's and C1b's boards with liars; the high end allows 5% (the largest boards cost 3% more).

In [ ]:
#| export
from explRL.bridge_plan import costRule20
from explRL.bridge_report import _d, _test
from explRL.clue_colony import checkOptions
from explRL.colony_grid import WORLD18, colonyCell
from explRL.colony_report import NAN, bootSeeds, cellVals
from explRL.relay_grid import ARMS23, N20
from explRL.relay_report import upperBound

ARMS24 = dict(bridge1=ARMS23['bridge1'], bridgeRole=ARMS23['bridgeRole'])
SEEDS24 = list(range(1810, 1818))                                   # fresh seeds, paired across the two arms
COST24, ALLOW24 = dict(bridge1=0.651, bridgeRole=0.662), 1.05      # $ per board with liars (C1's 8, C1b's 5 boards); allowance


def c1cCell(arm, seed, runner='llm'):
    "One C1c cell: study colony24, level '<arm>|independent', N 64, checked."
    c = colonyCell('colony24', arm, N20, seed, 'independent', runner, arms=ARMS24)
    checkOptions(c['cfg']['world'], N20, c['cfg'].get('nLiar', 0))
    return c


def c1cGrid(runner='llm'):
    "The 16 boards: each arm on SEEDS24."
    return [c1cCell(a, s, runner) for a in ARMS24 for s in SEEDS24]


def c1cRange(per=COST24, allow=ALLOW24):
    "(low, high) $ for c1cGrid: the measured cost per board, and that times the allowance."
    low = sum(per[c['level'].split('|')[0]] for c in c1cGrid())
    return round(low, 2), round(low * allow, 2)

In [ ]:
# test_c1c_grid (FR-1): 16 boards, fresh seeds, the two arms' options unchanged from C1b, and the estimate
g = c1cGrid()
assert len(g) == 16 and {c['level'] for c in g} == {'bridge1|independent', 'bridgeRole|independent'}
assert sorted({c['seed'] for c in g}) == SEEDS24 and not set(SEEDS24) & (set(range(1800, 1808)) | {1890})
assert all(c['N'] == 64 and c['cfg']['nLiar'] == 16 for c in g)
w = {c['level'].split('|')[0]: c['cfg']['world'] for c in g}
assert w['bridgeRole'].get('bridgeRole') and not w['bridgeRole'].get('evidence') and w['bridge1'].get('bridges') == 1
assert not w['bridge1'].get('bridgeRole') and c1cRange() == (10.5, 11.03)

## The plan

`C1C_PLAN` holds the gate, what it decides for A1, the calls recorded by hand before the run (as P23-10: pilot-only from C1b's 5 boards, and every-design-helps) and the cost rule. It was frozen on 2026-11-12, after the user approved the design and the estimate; its hash is pinned below.

In [ ]:
#| export
_G = 'bridgeRole - bridge1, N 64, independent liars, seeds 1810-1817 paired'
C1C_PLAN = dict(
    version=1, frozen=True, plan='specs/multiagent/phase24/', model='claude-haiku-5-5', world=WORLD18, turns=40,   # frozen 2026-11-12 (the user approved the design and the $10.50-11.03 estimate)
    channel='blackboard', memory='full', effort='low', arms=ARMS24, cells=dict(liars={a: SEEDS24 for a in ARMS24}), seeds=SEEDS24,
    fault='scripted ClueLiar in the first 16 slots (MirrorLiar: also on mirrors), guards the next 16',
    hypotheses=dict(
        H24_1=f'Gate C1c (i), effective: honestPooled, {_G}: one-sided exact sign-flip > 0, alpha 0.05 (smallest p 1/256)',
        H24_2=f'Gate C1c (ii), safe (non-inferiority): contamination, {_G}: the 95th percentile of 2,000 paired bootstrap means < 0.05'),
    gate=dict(parts=['H24_1', 'H24_2'], arm='bridgeRole', base='bridge1', alpha=0.05, metric='honestPooled', safety='contamination',
              margin=0.05, quantile=95, decides=dict(
                  passes="A1's bridged arms use a dedicated bridge without evidence (bridgeRole), in place of bridge1 (amends P19-8)",
                  fails='A1 keeps bridge1 (P19-8 as amended by C1); the dedicated role is not adopted')),
    robustness='part (i) on welfareHonest; an estimate of the opposite sign marks it not robust',
    why="Gate C1b asked (2026-11-12): bridgeFix - bridge1 -0.003 (p 0.62), safer; bridgeRole was the best coupled arm on C1b's 5 "
        "exploratory boards, chosen after seeing them, so the user asked for this confirmation before A1 adopts it",
    calls=dict(basis="C1b's 5 paired boards, seeds 1800-1804 (exploratory): bridgeRole beat bridge1 on all 5",
               pilotOnly=dict(H24_1=dict(call='passes', estimate=0.0207), H24_2=dict(call='holds', estimate=-0.1132), gate='passes'),
               everyDesignHelps=dict(H24_1=dict(call='passes'), H24_2=dict(call='holds'), gate='passes')),
    noPilot='both arms ran at this size in C1 and C1b with invalid replies under 0.2% (the user, 2026-11-12)',
    costRule=dict(rule='report and wait (P20-8): the user approved the estimate before the run', ceiling=12.0,
                  estimate=c1cRange(), perBoard=COST24, allowance=ALLOW24),
    stop='invalid replies >= 5% or the monthly cap: stop and ask', approved=dict(design='2026-11-12', estimate=c1cRange()),
    cap=250.0, capNote='per calendar month (the user, 2026-11-09)', dirs=dict(main='results/multiagent/colony/c1c'),
    nboot=2000, confirmatory=True)

In [ ]:
# test_c1c_plan (FR-2, FR-3): frozen with its hash pinned; the gate, the calls and the approval recorded
from explRL.coop_grid import coopHash
assert C1C_PLAN['frozen'] and coopHash(C1C_PLAN) == 'c7a01f678d5a', coopHash(C1C_PLAN)
assert C1C_PLAN['cap'] == 250.0 and 'calendar month' in C1C_PLAN['capNote'] and C1C_PLAN['gate']['margin'] == 0.05
assert set(C1C_PLAN['hypotheses']) == {'H24_1', 'H24_2'} and set(C1C_PLAN['gate']['decides']) == {'passes', 'fails'}
assert all(set(C1C_PLAN['hypotheses']) <= set(C1C_PLAN['calls'][k]) for k in ('pilotOnly', 'everyDesignHelps'))
assert C1C_PLAN['approved']['estimate'] == c1cRange() and c1cRange()[1] <= C1C_PLAN['costRule']['ceiling']

## The gate and the report

In [ ]:
#| export
def gateC1c(rows, plan=C1C_PLAN, nboot=2000, metric='honestPooled'):
    """Gate C1c over the 8 paired boards. Part (i): arm - base in `metric`, one-sided exact sign-flip, passes at p < alpha with a
    positive estimate. Part (ii): arm - base in contamination, holds when the 95th percentile of the paired bootstrap means is
    below the margin. Verdict: passes (both) or fails, with what it decides for A1."""
    G, s8 = plan['gate'], set(plan['seeds'])
    i = _test(_d(rows, G['arm'], G['base'], metric, seeds=s8), nboot)
    i['passes'] = bool(i['p'] < G['alpha'] and i['estimate'] > 0)
    c = _d(rows, G['arm'], G['base'], G['safety'], seeds=s8)
    ii = dict(_test(c, nboot, 'less'), upper=upperBound(c, nboot, G['quantile']), margin=G['margin'])
    ii['holds'] = bool(ii['upper'] < G['margin'])
    v = 'passes' if i['passes'] and ii['holds'] else 'fails'
    return dict(metric=metric, i=i, ii=ii, passes=v == 'passes', verdict=v, decides=G['decides'][v])


C1C_MEASURES = ('honestPooled', 'welfareHonest', 'contamination', 'relaysPerBridgeTurn', 'arrivalConfirmed', 'coupledContamination',
                'crossContamination', 'inTokPerCallBridge')


def c1cSummary(rows, plan=C1C_PLAN, nboot=None):
    """The gate with part (i)'s robustness on welfareHonest, each arm's measures (mean and bootstrap interval over its boards),
    and the hand-recorded calls scored against the verdicts. ValueError unless all 16 planned cells are present."""
    nboot, want = nboot or plan['nboot'], {(f'{a}|independent', s) for a, ss in plan['cells']['liars'].items() for s in ss}
    key = lambda r: (r['level'], int(float(r['seed'])))
    rows = [r for r in rows if int(float(r['N'])) == N20 and key(r) in want]
    if miss := sorted(want - {key(r) for r in rows}):
        raise ValueError(f'{len(miss)} planned C1c cells missing, e.g. {miss[:4]}')
    gate, rob = gateC1c(rows, plan, nboot), gateC1c(rows, plan, nboot, 'welfareHonest')
    gate['robust'], gate['welfareHonest'] = bool(rob['i']['estimate'] * gate['i']['estimate'] > 0), rob['i']
    arms = {a: {m: bootSeeds({k: v for k, v in cellVals(rows, a, 'independent', m, [N20]).items() if v == v}, nboot=nboot)
                for m in C1C_MEASURES} for a in plan['arms']}
    got = dict(H24_1='passes' if gate['i']['passes'] else 'fails', H24_2='holds' if gate['ii']['holds'] else 'fails', gate=gate['verdict'])
    calls = {k: {h: dict(call=v['call'] if isinstance(v, dict) else v, observed=got[h],
                         right=(v['call'] if isinstance(v, dict) else v) == got[h]) for h, v in plan['calls'][k].items()}
             for k in ('pilotOnly', 'everyDesignHelps')}
    cost = sum(float(r.get('cost', 0) or 0) for r in rows)
    return dict(gate=gate, arms=arms, calls=calls, boards=len(rows), cost=cost,
                invalid=sum(float(r['invalid']) for r in rows) / (sum(float(r['calls']) for r in rows) or 1))


def writeC1c(path, S, meta, plan=C1C_PLAN):
    "report.md: the gate, part (i)'s robustness, the arms side by side and the calls."
    g, f = S['gate'], lambda x: 'n/a' if x != x else f'{x:.3f}'
    ci = lambda t: f"[{f(t['ci'][0])}, {f(t['ci'][1])}]"
    L = [f"# Phase 24 (C1c): confirm the dedicated bridge (confirmatory)", '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, boards {S['boards']}, cost ${S['cost']:.2f}, invalid {S['invalid']:.4f}", '',
         f"## Gate C1c ({plan['gate']['arm']} - {plan['gate']['base']}, 64 agents, independent liars, seeds 1810-1817 paired)", '',
         '| item | boards | estimate | 95% interval | p (one-sided, exact) | verdict |', '|---|---|---|---|---|---|',
         f"| (i) effective: honestPooled | {g['i']['n']} | {f(g['i']['estimate'])} | {ci(g['i'])} | {g['i']['p']:.3f} | passes: {g['i']['passes']} |",
         f"| (i) robustness: welfareHonest | {g['welfareHonest']['n']} | {f(g['welfareHonest']['estimate'])} | {ci(g['welfareHonest'])} | "
         f"{g['welfareHonest']['p']:.3f} | same sign: {g['robust']} |",
         f"| (ii) safe: contamination (p for < 0) | {g['ii']['n']} | {f(g['ii']['estimate'])} | {ci(g['ii'])} | {g['ii']['p']:.3f} | "
         f"holds: {g['ii']['holds']} (95th percentile {f(g['ii']['upper'])} against {g['ii']['margin']}) |", '',
         f"**Gate C1c: {g['verdict']}.** {g['decides']}", '', '## Per arm (mean [95% interval] over 8 boards)', '',
         '| arm | ' + ' | '.join(C1C_MEASURES) + ' |', '|---|' + '---|' * len(C1C_MEASURES)]
    L += [f'| {a} | ' + ' | '.join(f'{f(v[0])} [{f(v[1])}, {f(v[2])}]' for v in S['arms'][a].values()) + ' |' for a in S['arms']]
    L += ['', '## Calls recorded before the run', '', '| line | item | call | observed | right |', '|---|---|---|---|---|']
    L += [f'| {k} | {h} | {c["call"]} | {c["observed"]} | {c["right"]} |' for k, cs in S['calls'].items() for h, c in cs.items()]
    open(path, 'w').write('\n'.join(L) + '\n')
    return '\n'.join(L)

In [ ]:
# test_c1c_gate (FR-4, FR-5): synthetic rows: a clear gain that is safe passes; no gain fails; a gain that is less safe fails;
# missing cells refuse; the report writes the verdict and scores the calls
import tempfile
from pathlib import Path


def rows24(gain, dcont, noise=0.004):
    out = []
    for a in ARMS24:
        for i, s in enumerate(SEEDS24):
            e = (i % 3 - 1) * noise
            hp = 0.18 + e + (gain if a == 'bridgeRole' else 0)
            out.append(dict(level=f'{a}|independent', seed=s, N=64, honestPooled=hp, welfareHonest=hp + 0.05, calls=1920.0, invalid=1.0,
                            contamination=0.33 + e + (dcont if a == 'bridgeRole' else 0), relaysPerBridgeTurn=0.1, arrivalConfirmed=0.5,
                            coupledContamination=0.7, crossContamination=0.01, inTokPerCallBridge=1100.0, cost=0.65))
    return out


S = c1cSummary(rows24(0.02, -0.1), nboot=200)
assert S['gate']['verdict'] == 'passes' and S['gate']['i']['p'] == 1 / 256 and S['gate']['robust'] and S['boards'] == 16
assert all(c['right'] for cs in S['calls'].values() for c in cs.values()) and abs(S['cost'] - 16 * 0.65) < 1e-9
assert gateC1c(rows24(0.0, -0.1), nboot=200)['verdict'] == 'fails'                      # no gain
G = gateC1c(rows24(0.02, 0.08), nboot=200)
assert G['i']['passes'] and not G['ii']['holds'] and G['verdict'] == 'fails'            # effective but less safe
assert gateC1c(rows24(-0.02, -0.1), nboot=200)['verdict'] == 'fails'                     # worse
S0 = c1cSummary(rows24(0.0, -0.1), nboot=200)
assert not S0['calls']['pilotOnly']['H24_1']['right'] and S0['calls']['pilotOnly']['H24_2']['right']
try:
    c1cSummary(rows24(0.02, -0.1)[1:]); assert False
except ValueError:
    pass
p = Path(tempfile.mkdtemp()) / 'report.md'
txt = writeC1c(p, S, dict(planHash='x', commit='y'))
assert 'Gate C1c: passes.' in txt and 'bridgeRole' in txt and p.read_text() == txt + '\n'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()